# TFM — 05. Interpretabilidad: factores de riesgo para prevención vial (Pregunta 2)
**Autora:** Meritxell Abellan Collado

Este notebook responde a la Pregunta 2 del TFM: *¿qué factores se asocian con una mayor gravedad, para informar campañas de prevención vial dirigidas?*

Usa el **modelo de referencia** ya entrenado y guardado en `04_Modelizacion.ipynb` (todas las variables, incluido el perfil de las personas implicadas) — no el modelo operativo, porque aquí el objetivo no es predecir en tiempo real sino entender qué pesa más en el riesgo de gravedad, y el modelo de referencia captura más señal (ver comparación Sección 3 de `04_Modelizacion.ipynb`).

Se usa **SHAP (SHapley Additive exPlanations)** para descomponer cada predicción en la contribución de cada variable, con dos niveles de lectura:
- **Global:** ¿qué variables son las más importantes en conjunto? (bar plot de |SHAP| medio, beeswarm plot)
- **Focalizado:** ¿cómo varía el riesgo según distrito, franja horaria y edad? — las tres dimensiones de focalización que se plantearon como objetivo práctico del TFM.


## 0. Carga del modelo de referencia y de los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap

import sys
sys.path.append('..')

pd.set_option('display.max_columns', 60)
RANDOM_STATE = 42
PATH = '../data/processed/features/'

X_train_ref = pd.read_csv(PATH + 'X_train_referencia.csv')
X_test_ref = pd.read_csv(PATH + 'X_test_referencia.csv')
y_train = pd.read_csv(PATH + 'y_train.csv').squeeze()
y_test = pd.read_csv(PATH + 'y_test.csv').squeeze()

modelo_referencia = joblib.load('../models/modelo_referencia.joblib')

print(f'Modelo cargado: {type(modelo_referencia).__name__}')
print(f'Referencia — train: {X_train_ref.shape} | test: {X_test_ref.shape}')


## 1. Cálculo de valores SHAP

Se calculan los valores SHAP sobre `X_test_ref` (no sobre train, para que la explicación refleje cómo se comporta el modelo sobre datos que no ha visto, coherente con el resto del proyecto). Con ~10.678 filas de test, se usa una muestra para que el cálculo sea manejable en tiempo razonable — `shap.Explainer` con un modelo basado en árboles (Gradient Boosting/Random Forest) es eficiente, pero aun así conviene acotar el tamaño de muestra para el cálculo exacto.


In [ ]:
N_MUESTRA_SHAP = 2000  # ajustar si el tiempo de cálculo lo permite

muestra_shap = X_test_ref.sample(n=min(N_MUESTRA_SHAP, len(X_test_ref)), random_state=RANDOM_STATE)

explainer = shap.Explainer(modelo_referencia, X_train_ref)
shap_values = explainer(muestra_shap)

print(f'Valores SHAP calculados sobre {len(muestra_shap):,} accidentes de test')


## 2. Importancia global de variables

**Bar plot:** ranking de variables por magnitud media absoluta de su contribución SHAP — cuánto pesa cada variable en promedio, sin importar si empuja el riesgo hacia arriba o hacia abajo.


In [ ]:
shap.plots.bar(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig('../figures/15_shap_importancia_global.png', dpi=150, bbox_inches='tight')
plt.show()


**Beeswarm plot:** cada punto es un accidente; el color indica el valor de la variable (rojo=alto, azul=bajo) y la posición horizontal indica si esa variable empujó la predicción hacia mayor gravedad (derecha) o menor (izquierda). Permite ver no solo qué variables importan, sino en qué dirección.


In [ ]:
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig('../figures/16_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Focalización por distrito, franja horaria y edad

Estas son las tres dimensiones de focalización planteadas como objetivo práctico del TFM (campañas de prevención dirigidas). Se examina la contribución SHAP de las variables correspondientes a cada una.


In [ ]:
# Distrito: TASA_GRAVEDAD_HIST_DISTRITO — su valor SHAP indica cuánto pesa
# el historial de gravedad de cada distrito en la predicción de un accidente concreto
idx_distrito = list(muestra_shap.columns).index('TASA_GRAVEDAD_HIST_DISTRITO')
shap.plots.scatter(shap_values[:, idx_distrito], show=False)
plt.title('Contribución SHAP de la tasa histórica de gravedad del distrito')
plt.tight_layout()
plt.savefig('../figures/17_shap_distrito.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Franja horaria: combinar la contribución de HORA_SIN y HORA_COS
# (la codificación cíclica reparte la señal horaria entre las dos columnas)
idx_hora_sin = list(muestra_shap.columns).index('HORA_SIN')
idx_hora_cos = list(muestra_shap.columns).index('HORA_COS')

contribucion_hora = shap_values.values[:, idx_hora_sin] + shap_values.values[:, idx_hora_cos]
hora_aprox = np.arctan2(muestra_shap['HORA_SIN'], muestra_shap['HORA_COS']) * 24 / (2 * np.pi)
hora_aprox = hora_aprox.where(hora_aprox >= 0, hora_aprox + 24).round().astype(int)

tabla_hora = pd.DataFrame({'HORA': hora_aprox, 'contribucion_shap': contribucion_hora})
resumen_hora = tabla_hora.groupby('HORA')['contribucion_shap'].mean().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
colores = ['#2a78d6' if v > 0 else '#c3c2b7' for v in resumen_hora]
ax.bar(resumen_hora.index, resumen_hora.values, color=colores)
ax.axhline(0, color='grey', linewidth=1)
ax.set_xlabel('Hora del día')
ax.set_ylabel('Contribución SHAP media (a mayor gravedad)')
ax.set_title('Contribución de la franja horaria a la predicción de gravedad')
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.savefig('../figures/18_shap_hora.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Edad de riesgo: INCLUYE_EDAD_RIESGO
idx_edad = list(muestra_shap.columns).index('INCLUYE_EDAD_RIESGO')
shap.plots.scatter(shap_values[:, idx_edad], show=False)
plt.title('Contribución SHAP de INCLUYE_EDAD_RIESGO')
plt.tight_layout()
plt.savefig('../figures/19_shap_edad_riesgo.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Conclusiones para el diseño de campañas de prevención vial

*(completar con la lectura de los gráficos anteriores una vez ejecutados: qué distritos concentran mayor contribución positiva al riesgo, qué franjas horarias destacan, y si `INCLUYE_EDAD_RIESGO` empuja consistentemente hacia mayor gravedad — esto responde directamente a la Pregunta 2 y debe trasladarse a la memoria final como recomendaciones concretas de focalización geográfica, temporal y demográfica.)*
